# Lab 11: Logistic Regression and Classification
> Week 11 | CLO3 | ISLP Ch.4.1–4.3

## บทนำสัปดาห์

สัปดาห์นี้เราจะก้าวจาก Regression มาสู่ **Classification** ซึ่งเป็น supervised learning อีกประเภทหนึ่งที่ response variable Y เป็น categorical แทนที่จะเป็น continuous Logistic Regression เป็น classifier พื้นฐานที่ Data Scientist ทุกคนต้องรู้จัก เพราะมันเป็นทั้ง interpretable (ตีความ coefficient ได้ชัดเจน) และ effective (ทำงานได้ดีสำหรับ linear decision boundary) ใน lab นี้คุณจะสร้าง Logistic Regression, ดู sigmoid curve, ตีความ z-statistic และ Odds Ratio, สร้าง confusion matrix และคำนวณ Precision/Recall/F1 เพื่อประเมิน model อย่างครบถ้วน

**LLo**: อธิบายข้อจำกัดของ Linear Regression สำหรับ Classification และสร้าง Logistic Regression model ได้

**สิ่งที่จะเรียนรู้**:
- Part 1: Why not Linear Regression? — OLS ให้ predictions นอก [0,1]
- Part 2: Simple Logistic Regression — sigmoid, log-odds, MLE
- Part 3: Multiple Logistic Regression — confounding example
- Part 4: Classification Evaluation — confusion matrix, metrics
- Part 5: Case Study — Credit fraud detection


In [ ]:
# ─── Import libraries ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                              ConfusionMatrixDisplay, accuracy_score,
                              precision_score, recall_score, f1_score)
from scipy.special import expit  # sigmoid function

plt.rcParams['figure.figsize'] = (9, 5)
np.random.seed(42)
print('Libraries loaded ✓')

In [ ]:
# ─── โหลด Default dataset ──────────────────────────────────────────
# วัตถุประสงค์: ใช้ข้อมูล credit default 10,000 คน
try:
    default = pd.read_csv('https://www.statlearning.com/s/Default.csv', index_col=0)
    default.columns = default.columns.str.lower()
except:
    # Synthetic Default dataset
    np.random.seed(0)
    n = 10000
    balance = np.random.exponential(scale=900, size=n)
    balance = np.clip(balance, 0, 3000)
    income  = np.random.normal(35000, 15000, n)
    student = np.random.choice([0, 1], n, p=[0.7, 0.3])
    log_odds = -10.65 + 0.0055*balance - 0.000002*income - 0.65*student
    prob_default = expit(log_odds)
    default_y = np.random.binomial(1, prob_default)
    default = pd.DataFrame({'default': ['Yes' if d else 'No' for d in default_y],
                            'student': ['Yes' if s else 'No' for s in student],
                            'balance': balance, 'income': income})

# Encode binary columns
default['default_num'] = (default['default'] == 'Yes').astype(int)
default['student_num'] = (default['student'] == 'Yes').astype(int)

print('Default dataset shape:', default.shape)
print('Default rate:', default['default'].value_counts(normalize=True).round(4))
default.head()

---
## Part 1: Why Not Linear Regression?

**Part นี้เราจะแสดงว่า OLS ให้ predictions นอก [0,1]** เพื่อเป็น motivation ว่าทำไมต้องใช้ Logistic Regression

เราจะ fit OLS บน binary outcome แล้วดูว่าบาง predictions ออกนอกช่วง [0,1]


In [ ]:
# ─── OLS บน binary outcome ──────────────────────────────────────────
# วัตถุประสงค์: แสดงว่า OLS ให้ probability ที่ไม่ valid (นอก [0,1])
m_ols = smf.ols('default_num ~ balance', data=default).fit()
ols_pred = m_ols.fittedvalues

print(f'OLS predictions outside [0,1]: {((ols_pred < 0) | (ols_pred > 1)).sum()} cases')
print(f'Min prediction: {ols_pred.min():.4f}')
print(f'Max prediction: {ols_pred.max():.4f}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bal = np.linspace(default['balance'].min(), default['balance'].max(), 200)

# OLS
ols_line = m_ols.params['Intercept'] + m_ols.params['balance'] * bal
axes[0].scatter(default['balance'], default['default_num'], alpha=0.03, s=5)
axes[0].plot(bal, ols_line, 'r-', lw=2)
axes[0].axhline(0, color='gray', lw=0.8, linestyle='--')
axes[0].axhline(1, color='gray', lw=0.8, linestyle='--')
axes[0].set_title('OLS: predictions can be < 0 or > 1!')
axes[0].set_xlabel('Balance'); axes[0].set_ylabel('P(default=Yes)')

# Logistic
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000).fit(default[['balance']], default['default_num'])
log_prob = lr.predict_proba(bal.reshape(-1,1))[:,1]
axes[1].scatter(default['balance'], default['default_num'], alpha=0.03, s=5)
axes[1].plot(bal, log_prob, 'b-', lw=2)
axes[1].axhline(0, color='gray', lw=0.8, linestyle='--')
axes[1].axhline(1, color='gray', lw=0.8, linestyle='--')
axes[1].set_title('Logistic: always in [0,1] ✓')
axes[1].set_xlabel('Balance'); axes[1].set_ylabel('P(default=Yes)')

plt.tight_layout(); plt.show()

---
## Part 2: Simple Logistic Regression

**Part นี้เราจะ fit Logistic Regression** และดู sigmoid curve, z-statistic, p-value และ decision boundary


In [ ]:
# ─── Fit Logistic Regression ด้วย statsmodels ──────────────────────
# วัตถุประสงค์: ได้ full inference (z-stat, p-value, CI) เหมือน regression summary
m_logit = smf.logit('default_num ~ balance', data=default).fit()
print(m_logit.summary())

In [ ]:
# ─── Verify z = β̂ / SE ────────────────────────────────────────────
# วัตถุประสงค์: แสดงว่า z-statistic คำนวณเหมือน t-statistic แต่ใช้ N(0,1)
beta1 = m_logit.params['balance']
se1   = m_logit.bse['balance']
z_manual = beta1 / se1
print(f'β̂₁ = {beta1:.6f}')
print(f'SE = {se1:.6f}')
print(f'z  = β̂₁/SE = {z_manual:.4f}')
print(f'Check: model z = {m_logit.tvalues["balance"]:.4f} ✓')

# Odds Ratio สำหรับ $1,000 เพิ่ม
OR_1000 = np.exp(1000 * beta1)
print(f'\nOdds Ratio (balance +$1000): {OR_1000:.4f}')
print(f'→ Odds of default multiply by {OR_1000:.2f} per $1,000 increase in balance')

In [ ]:
# ─── Sigmoid curve + decision boundary ─────────────────────────────
# วัตถุประสงค์: visualize model — sigmoid curve + จุด X* ที่ P = 0.5
bal_range = np.linspace(0, 3000, 300)
prob_range = m_logit.predict(pd.DataFrame({'balance': bal_range}))

# Decision boundary: β₀ + β₁x = 0 → x = -β₀/β₁
x_star = -m_logit.params['Intercept'] / m_logit.params['balance']
print(f'Decision boundary: balance = ${x_star:.0f}')

plt.figure(figsize=(9, 5))
colors = ['red' if d else 'steelblue' for d in default['default_num']]
plt.scatter(default['balance'], default['default_num'], alpha=0.04, s=5, c=colors)
plt.plot(bal_range, prob_range, 'k-', lw=2.5, label='P(default=Yes|balance)')
plt.axhline(0.5, color='orange', lw=1.5, linestyle='--', label='τ=0.5')
plt.axvline(x_star, color='green', lw=1.5, linestyle='--', label=f'Decision boundary (${x_star:.0f})')
plt.xlabel('Credit Card Balance ($)')
plt.ylabel('P(Default = Yes)')
plt.title('Logistic Regression: P(Default) vs Balance')
plt.legend(); plt.tight_layout(); plt.show()

### TODO 1 (Easy): Predict สำหรับ Balance ที่กำหนด

**สิ่งที่ต้องทำ**:
1. Predict P(default) สำหรับ balance = $1,000, $2,000, $3,000
2. ระบุว่าแต่ละกรณี predict class ใด (threshold = 0.5)
3. อธิบาย: ลูกค้าที่มี balance ต่างกัน $1,000 (เช่น $1,000 vs $2,000) — P(default) ต่างกันแค่ไหน?


In [ ]:
# TODO 1: Predict for specific balance values
# เติม code ที่นี่

---
## Part 3: Multiple Logistic Regression + Confounding

**Part นี้เราจะเพิ่ม predictors เข้า model** และสังเกต confounding effect ที่โด่งดัง: student status ดูเหมือน "bad" ใน SLR แต่กลับ "good" ใน MLR เมื่อ control balance


In [ ]:
# ─── SLR: Default ~ Student ─────────────────────────────────────────
# วัตถุประสงค์: ดู coefficient ของ student ก่อน control balance
m_slr_student = smf.logit('default_num ~ student_num', data=default).fit(disp=False)
print('SLR: Default ~ Student')
print(f'  β̂_student = {m_slr_student.params["student_num"]:.4f}')
print(f'  OR_student = {np.exp(m_slr_student.params["student_num"]):.4f}')
print(f'  p-value   = {m_slr_student.pvalues["student_num"]:.4f}')

# ─── MLR: Default ~ Balance + Income + Student ──────────────────────
# วัตถุประสงค์: ดู coefficient ของ student หลัง control balance
m_mlr = smf.logit('default_num ~ balance + income + student_num', data=default).fit(disp=False)
print('\nMLR: Default ~ Balance + Income + Student')
print(m_mlr.summary2().tables[1][['Coef.','Std.Err.','z','P>|z|']].round(6))

# Confounding visualization
fig, ax = plt.subplots(figsize=(7, 4))
default.groupby('student')['balance'].plot(kind='density', ax=ax, legend=True)
ax.set_title('Balance Distribution: Students vs Non-Students')
ax.set_xlabel('Balance')
plt.tight_layout(); plt.show()
print('Students have higher average balance → explains confounding!')

### TODO 2 (Medium): สร้างตาราง Odds Ratios และ Predict

**สิ่งที่ต้องทำ**:
1. สร้าง DataFrame แสดง β̂, OR = e^β̂, 95% CI ของ OR สำหรับ M2 ทุก predictor
2. Predict P(default) สำหรับ 3 customers: (1000, 40000, No), (2500, 70000, Yes), (500, 25000, No)
3. Plot: P(default) vs balance แยกสี student/non-student (เส้น 2 เส้น)


In [ ]:
# TODO 2: Odds Ratios + Predictions
# เติม code ที่นี่

---
## Part 4: Classification Evaluation

**Part นี้เราจะประเมิน classifier** โดยใช้ confusion matrix และ metrics ต่าง ๆ

ความสำคัญ: ธนาคารต้องการ recall สูง (จับ default ได้มาก) มากกว่า precision — ต้องปรับ threshold


In [ ]:
# ─── Train-test split ──────────────────────────────────────────────
# วัตถุประสงค์: ประเมิน model บน test set ที่ไม่ได้ใช้ train
X = default[['balance', 'income', 'student_num']]
y = default['default_num']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit sklearn Logistic Regression (สะดวกสำหรับ prediction)
lr_final = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred   = lr_final.predict(X_test)
y_proba  = lr_final.predict_proba(X_test)[:, 1]

print(f'Test set size: {len(y_test)}')
print(f'Actual defaults in test: {y_test.sum()}')

In [ ]:
# ─── Confusion Matrix + Metrics ─────────────────────────────────────
# วัตถุประสงค์: ดู TP, TN, FP, FN และ metrics ที่สำคัญ
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['No Default','Default']).plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix (threshold=0.5)')
plt.tight_layout(); plt.show()

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['No Default','Default']))

### TODO 3 (Medium): Threshold Analysis

**สิ่งที่ต้องทำ**:
1. ลอง threshold = 0.2, 0.3, 0.5 เปรียบ Precision, Recall, F1
2. Plot: Precision และ Recall vs threshold (x-axis)
3. แนะนำ threshold ที่เหมาะสำหรับ "ธนาคารต้องจับ default ให้ได้มากที่สุด"


In [ ]:
# TODO 3: Threshold analysis
# เติม code ที่นี่

---
## Part 5: Case Study — E-Commerce Fraud Detection

**Scenario**: บริษัท E-Commerce ต้องการตรวจจับ fraudulent transactions (1 = fraud, 0 = legitimate)

Dataset: synthetic fraud data — 50,000 transactions, fraud rate ≈ 5%


In [ ]:
# ─── Fraud Detection Dataset ───────────────────────────────────────
# วัตถุประสงค์: ตัวอย่างจริงของ binary classification ที่ imbalanced
np.random.seed(777)
n = 50000
amount    = np.random.exponential(100, n)  # transaction amount
is_night  = np.random.binomial(1, 0.3, n)  # กลางคืน
new_device= np.random.binomial(1, 0.2, n)  # device ใหม่
# fraud probability
log_odds_fraud = (-4 + 0.005*amount + 1.5*is_night + 2*new_device)
p_fraud = expit(log_odds_fraud)
fraud = np.random.binomial(1, p_fraud)

df_fraud = pd.DataFrame({'amount': amount, 'is_night': is_night,
                         'new_device': new_device, 'fraud': fraud})
print(f'Fraud rate: {fraud.mean():.4f} ({fraud.sum()} / {n})')

### TODO 4 (Hard): Fraud Detection — Full Pipeline

**สิ่งที่ต้องทำ**:
1. Fit Logistic Regression: `fraud ~ amount + is_night + new_device`
2. Train/test split (test=20%)
3. Confusion matrix และ classification report
4. เพราะ fraud rare (imbalanced) — ใช้ `class_weight='balanced'` เปรียบ recall ก่อน-หลัง
5. หา threshold ที่ให้ recall ≥ 0.8 สำหรับ fraud class
6. เขียน Business Report 5 ประโยค: "บริษัทสามารถตรวจ fraud ได้ ___% ของ fraud ทั้งหมด ด้วย FP rate ___..."


In [ ]:
# TODO 4: Fraud Detection Pipeline
# วัตถุประสงค์: ฝึก full pipeline ที่ใช้จริงใน fraud detection
# เติม code ที่นี่

---
## สรุป Lab 11

| Concept | สูตร | Python |
|---------|------|--------|
| Sigmoid | 1/(1+e⁻ᶻ) | `scipy.special.expit(z)` |
| Log-odds | log(p/(1-p)) = β₀+β₁X | `smf.logit('y~x').fit()` |
| z-statistic | β̂/SE | `model.tvalues` |
| Odds Ratio | e^β̂ | `np.exp(model.params)` |
| Predict proba | — | `model.predict_proba(X)[:,1]` |
| Confusion matrix | TP/TN/FP/FN | `confusion_matrix(y,ŷ)` |
| Precision | TP/(TP+FP) | `precision_score(y,ŷ)` |
| Recall | TP/(TP+FN) | `recall_score(y,ŷ)` |
| F1 | 2PR/(P+R) | `f1_score(y,ŷ)` |

## Reflection Questions

1. **Threshold choice**: ถ้าคุณเป็น Data Scientist ให้ธนาคาร — คุณจะเลือก threshold เท่าไรสำหรับ credit default detection? ทำไม? มี trade-off อะไรบ้าง?

2. **Logistic vs OLS**: ถ้า default rate เป็น 50% (balanced) — OLS และ Logistic Regression จะให้ decision boundary ที่ต่างกันมากไหม? ทำไม?

3. **Confounding**: ในชีวิตจริง มีตัวอย่างอื่น ๆ ที่ coefficient ของ predictor เปลี่ยน sign เมื่อเพิ่ม predictor อื่นไหม? ยกตัวอย่าง 1 ตัวอย่าง
